In [7]:
# --- Cell 1: setup (paths, config, peers) ---
from pathlib import Path
import datetime as dt
import pandas as pd
import numpy as np
import yaml, os

ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
CFG  = ROOT / "project" / "config" / "model_v2.yml"

with open(CFG, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

RAW_DIR       = ROOT / cfg["paths"]["raw_dir"]
PROCESSED_DIR = ROOT / cfg["paths"]["processed_dir"]
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

PEERS = cfg["project"]["peers"]  # names you set in YAML
BANK_TICKERS = {
    "JP Morgan":       "JPM",
    "Deutsche Bank":   "DB",
    "Morgan Stanley":  "MS",
    "Bank of America": "BAC",
    "Goldman Sachs":   "GS",
}

def ts():
    return dt.datetime.now().strftime("%Y%m%d-%H%M")


In [8]:
# --- Cell 2: load API keys (.env) ---
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

FMP_API_KEY = os.getenv("FMP_API_KEY", "").strip()  # Financial Modeling Prep


In [13]:
FMP_API_KEY= "khushi_123"

In [14]:
# --- Cell 3: helpers ---
import requests

def http_get_json(url, params=None, timeout=30):
    r = requests.get(url, params=params or {}, timeout=timeout)
    r.raise_for_status()
    return r.json()

def to_quarter_end(dates):
    # Accept str or Timestamp; return Timestamp at quarter-end
    s = pd.to_datetime(dates)
    return pd.PeriodIndex(s, freq="Q").to_timestamp("Q")

def save_raw_csv(df: pd.DataFrame, kind: str, tag: str):
    path = RAW_DIR / f"{kind}_{tag}_{ts()}.csv"
    df.to_csv(path, index=False)
    print("Saved:", path)
    return path

def validate_cols(df, req):
    miss = [c for c in req if c not in df.columns]
    assert not miss, f"Missing columns: {miss}"


In [15]:
# --- Cell 4: pull quarterly revenues via FMP (primary) ---
def fetch_revenue_fmp(ticker: str, limit=40):
    base = f"https://financialmodelingprep.com/api/v3/income-statement/{ticker}"
    params = {"period": "quarter", "limit": limit, "apikey": FMP_API_KEY}
    js = http_get_json(base, params=params)
    if not isinstance(js, list) or len(js) == 0:
        raise ValueError(f"Empty response for {ticker} income-statement")
    df = pd.DataFrame(js)
    # Different fields exist; prefer 'revenue', fallback to 'totalRevenue' / 'netRevenue'
    candidates = [c for c in ["revenue","totalRevenue","netRevenue"] if c in df.columns]
    if not candidates:
        raise ValueError(f"No revenue field found for {ticker}")
    rev_col = candidates[0]
    out = df[["date", rev_col]].copy()
    out["date"] = to_quarter_end(out["date"])
    out = out.rename(columns={rev_col: "revenue_total"})
    out["ticker"] = ticker
    return out

def collect_revenue_fmp(bank_to_ticker: dict):
    frames = []
    for bank, t in bank_to_ticker.items():
        try:
            df = fetch_revenue_fmp(t)
            df["bank"] = bank
            frames.append(df[["bank","ticker","date","revenue_total"]])
        except Exception as e:
            print(f"[FMP revenue] {bank} ({t}) -> {e}")
    if not frames:
        raise RuntimeError("FMP revenue fetch returned no data for all banks.")
    res = pd.concat(frames, ignore_index=True).drop_duplicates()
    return res.sort_values(["bank","date"])


In [16]:
# --- Cell 5: yfinance fallback for revenue ---
import warnings
warnings.filterwarnings("ignore")

try:
    import yfinance as yf
except ImportError:
    yf = None
    print("yfinance not installed; run: pip install yfinance")

def fetch_revenue_yf(ticker: str):
    if yf is None:
        raise RuntimeError("yfinance not available.")
    t = yf.Ticker(ticker)
    qf = t.quarterly_financials  # index: line items; columns: periods
    if qf is None or qf.empty:
        raise ValueError("Empty quarterly_financials")
    df = qf.T  # now rows = periods, columns = line items
    # Try common labels
    candidates = [c for c in ["Total Revenue","TotalRevenue","Operating Revenue","Revenue"] if c in df.columns]
    if not candidates:
        raise ValueError("No revenue column found in yfinance quarterly_financials")
    rev_col = candidates[0]
    out = df[[rev_col]].reset_index().rename(columns={"index":"date", rev_col:"revenue_total"})
    out["date"] = to_quarter_end(out["date"])
    out["ticker"] = ticker
    return out[["date","revenue_total","ticker"]]

def collect_revenue_yf(bank_to_ticker: dict):
    frames = []
    for bank, t in bank_to_ticker.items():
        try:
            df = fetch_revenue_yf(t)
            df["bank"] = bank
            frames.append(df[["bank","ticker","date","revenue_total"]])
        except Exception as e:
            print(f"[YF revenue] {bank} ({t}) -> {e}")
    if not frames:
        raise RuntimeError("yfinance revenue fetch returned no data for all banks.")
    res = pd.concat(frames, ignore_index=True).drop_duplicates()
    return res.sort_values(["bank","date"])


In [17]:
# --- Cell 6: execute revenue pull and save ---
if FMP_API_KEY:
    try:
        rev = collect_revenue_fmp(BANK_TICKERS)
        src = "fmp"
    except Exception as e:
        print("[FMP] Falling back due to:", e)
        rev = collect_revenue_yf(BANK_TICKERS)
        src = "yfinance"
else:
    print("No FMP_API_KEY found; using yfinance fallback.")
    rev = collect_revenue_yf(BANK_TICKERS)
    src = "yfinance"

validate_cols(rev, ["bank","ticker","date","revenue_total"])
rev_path = save_raw_csv(rev, kind=f"revenue_actuals_{src}", tag="banks5_quarterly")
rev.tail()


[FMP revenue] JP Morgan (JPM) -> 401 Client Error: Unauthorized for url: https://financialmodelingprep.com/api/v3/income-statement/JPM?period=quarter&limit=40&apikey=khushi_123
[FMP revenue] Deutsche Bank (DB) -> 401 Client Error: Unauthorized for url: https://financialmodelingprep.com/api/v3/income-statement/DB?period=quarter&limit=40&apikey=khushi_123
[FMP revenue] Morgan Stanley (MS) -> 401 Client Error: Unauthorized for url: https://financialmodelingprep.com/api/v3/income-statement/MS?period=quarter&limit=40&apikey=khushi_123
[FMP revenue] Bank of America (BAC) -> 401 Client Error: Unauthorized for url: https://financialmodelingprep.com/api/v3/income-statement/BAC?period=quarter&limit=40&apikey=khushi_123
[FMP revenue] Goldman Sachs (GS) -> 401 Client Error: Unauthorized for url: https://financialmodelingprep.com/api/v3/income-statement/GS?period=quarter&limit=40&apikey=khushi_123
[FMP] Falling back due to: FMP revenue fetch returned no data for all banks.
Saved: C:\Users\User\boot

,bank,ticker,date,revenue_total
17,Morgan Stanley,MS,2024-06-30,1.402400e+10
16,Morgan Stanley,MS,2024-09-30,1.433900e+10
15,Morgan Stanley,MS,2024-12-31,1.504300e+10
14,Morgan Stanley,MS,2025-03-31,1.651700e+10
13,Morgan Stanley,MS,2025-06-30,1.560400e+10


In [18]:
from pathlib import Path
import pandas as pd, numpy as np, datetime as dt, yaml

ROOT = Path.cwd().resolve().parents[1] if (Path.cwd().name == "notebooks") else Path.cwd()
CFG  = ROOT / "project" / "config" / "model_v2.yml"
with open(CFG, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

RAW_DIR = ROOT / cfg["paths"]["raw_dir"]
RAW_DIR.mkdir(parents=True, exist_ok=True)

def to_quarter_end(s):
    return pd.PeriodIndex(pd.to_datetime(s), freq="Q").to_timestamp("Q")

def ts():
    return dt.datetime.now().strftime("%Y%m%d-%H%M")


In [20]:
data_ms = [
    ["Morgan Stanley","MS","2024-06-30",1.402400e10],
    ["Morgan Stanley","MS","2024-09-30",1.433900e10],
    ["Morgan Stanley","MS","2024-12-31",1.504300e10],
    ["Morgan Stanley","MS","2025-03-31",1.651700e10],
    ["Morgan Stanley","MS","2025-06-30",1.560400e10],
]
rev_ms = pd.DataFrame(data_ms, columns=["bank","ticker","date","revenue_total"])


In [21]:
# Parse dates to quarter-end, enforce types, dedupe
rev_ms["date"] = to_quarter_end(rev_ms["date"])
rev_ms["revenue_total"] = pd.to_numeric(rev_ms["revenue_total"], errors="coerce")

# Basic checks
assert (rev_ms["revenue_total"] >= 0).all(), "Revenue must be non-negative"
assert rev_ms[["bank","date"]].duplicated().sum() == 0, "Duplicates per (bank,date) found"

# Sort + save a bank-specific file so we can union later
rev_ms = rev_ms.sort_values(["bank","date"]).reset_index(drop=True)
out = RAW_DIR / f"revenue_actuals_manual_MS_{ts()}.csv"
rev_ms.to_csv(out, index=False)
print("Saved:", out)
rev_ms


Saved: C:\Users\User\bootcamp_Khushi_Khanna\project\data\raw\revenue_actuals_manual_MS_20250825-1323.csv


,bank,ticker,date,revenue_total
0,Morgan Stanley,MS,2024-06-30,1.402400e+10
1,Morgan Stanley,MS,2024-09-30,1.433900e+10
2,Morgan Stanley,MS,2024-12-31,1.504300e+10
3,Morgan Stanley,MS,2025-03-31,1.651700e+10
4,Morgan Stanley,MS,2025-06-30,1.560400e+10


In [23]:
# --- Create JPMorgan revenue DataFrame ---
data_jpm = [
    ["JP Morgan","JPM","2024-06-30", <value_here>],
    ["JP Morgan","JPM","2024-09-30", <value_here>],
    ["JP Morgan","JPM","2024-12-31", <value_here>],
    ["JP Morgan","JPM","2025-03-31", <value_here>],
    ["JP Morgan","JPM","2025-06-30", <value_here>],
]
rev_jpm = pd.DataFrame(data_jpm, columns=["bank","ticker","date","revenue_total"])


SyntaxError: invalid syntax (1551999532.py, line 3)

In [22]:
# Parse dates to quarter-end, enforce types, dedupe
rev_jpm["date"] = to_quarter_end(rev_jpm["date"])
rev_jpm["revenue_total"] = pd.to_numeric(rev_ms["revenue_total"], errors="coerce")

# Basic checks
assert (rev_ms["revenue_total"] >= 0).all(), "Revenue must be non-negative"
assert rev_ms[["bank","date"]].duplicated().sum() == 0, "Duplicates per (bank,date) found"

# Sort + save a bank-specific file so we can union later
rev_jpm = rev_jpm.sort_values(["bank","date"]).reset_index(drop=True)
out = RAW_DIR / f"revenue_actuals_manual_JPM_{ts()}.csv"
rev_jpm.to_csv(out, index=False)
print("Saved:", out)
rev_jpm


NameError: name 'rev_jpm' is not defined

In [24]:
# --- Unified revenue pull for all 5 banks with FMP -> yfinance fallback, plus diagnostics ---

import os, datetime as dt
import pandas as pd
import numpy as np
from pathlib import Path

# ---- config/paths ----
ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
CFG  = ROOT / "project" / "config" / "model_v2.yml"
import yaml
with open(CFG, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
RAW_DIR = ROOT / cfg["paths"]["raw_dir"]
RAW_DIR.mkdir(parents=True, exist_ok=True)

PEERS = {
    "JP Morgan":       "JPM",
    "Deutsche Bank":   "DB",
    "Morgan Stanley":  "MS",
    "Bank of America": "BAC",
    "Goldman Sachs":   "GS",
}

def ts(): return dt.datetime.now().strftime("%Y%m%d-%H%M")
def to_quarter_end(d): return pd.PeriodIndex(pd.to_datetime(d), freq="Q").to_timestamp("Q")

def save_bank_csv(df, bank, ticker):
    df = df.copy()
    df["bank"] = bank
    df["ticker"] = ticker
    df = df[["bank","ticker","date","revenue_total"]].sort_values("date")
    out = RAW_DIR / f"revenue_actuals_manual_{ticker}_{ts()}.csv"
    df.to_csv(out, index=False)
    print(f"Saved → {out}")
    return out

def pretty(df, n=6):
    # small helper to show last n rows with billions formatting for readability
    d = df.copy()
    d["revenue_total_bil"] = d["revenue_total"] / 1e9
    return d.sort_values("date").tail(n)

# ---- FMP primary ----
FMP_API_KEY = os.getenv("FMP_API_KEY", "").strip()

import requests
def http_get_json(url, params=None, timeout=30):
    r = requests.get(url, params=params or {}, timeout=timeout)
    r.raise_for_status()
    return r.json()

def fetch_revenue_fmp(ticker: str, limit=40):
    base = f"https://financialmodelingprep.com/api/v3/income-statement/{ticker}"
    params = {"period": "quarter", "limit": limit, "apikey": FMP_API_KEY}
    js = http_get_json(base, params=params)
    df = pd.DataFrame(js)
    # choose the best revenue-like field present
    for col in ["revenue","totalRevenue","netRevenue"]:
        if col in df.columns:
            out = df[["date", col]].rename(columns={col: "revenue_total"}).copy()
            out["date"] = to_quarter_end(out["date"])
            out["revenue_total"] = pd.to_numeric(out["revenue_total"], errors="coerce")
            out = out.dropna(subset=["revenue_total"]).drop_duplicates(subset=["date"]).sort_values("date")
            if not out.empty:
                return out
    raise ValueError("FMP: no revenue column found")

# ---- yfinance fallback ----
try:
    import yfinance as yf
except Exception as e:
    yf = None
    print("yfinance not available; run: pip install yfinance")

def fetch_revenue_yf(ticker: str):
    if yf is None:
        raise RuntimeError("yfinance not installed")
    t = yf.Ticker(ticker)
    qf = t.quarterly_financials
    if qf is None or qf.empty:
        # Sometimes quarterly is empty; try annual to at least confirm structure
        ann = t.financials
        raise ValueError("yfinance: quarterly_financials empty")
    df = qf.T  # rows: periods, cols: line items
    # probe possible revenue column names
    for col in ["Total Revenue","TotalRevenue","Operating Revenue","Revenue"]:
        if col in df.columns:
            out = df[[col]].reset_index().rename(columns={"index":"date", col:"revenue_total"})
            out["date"] = to_quarter_end(out["date"])
            out["revenue_total"] = pd.to_numeric(out["revenue_total"], errors="coerce")
            out = out.dropna(subset=["revenue_total"]).drop_duplicates(subset=["date"]).sort_values("date")
            if not out.empty:
                return out
    raise ValueError("yfinance: no suitable revenue column found")

# ---- run for all banks ----
results = {}
for bank, ticker in PEERS.items():
    df = None
    err_primary = err_fallback = None

    if FMP_API_KEY:
        try:
            df = fetch_revenue_fmp(ticker)
            source = "FMP"
        except Exception as e:
            err_primary = str(e)

    if df is None:
        try:
            df = fetch_revenue_yf(ticker)
            source = "yfinance"
        except Exception as e:
            err_fallback = str(e)

    if df is None or df.empty:
        print(f"❌ {bank} ({ticker}) — could not fetch.\n  FMP err: {err_primary}\n  YF err: {err_fallback}\n"
              "  Tip: You can paste a small manual list of [date, revenue_total] and we’ll save a CSV.\n")
        continue

    # Basic QA
    df = df[["date","revenue_total"]].dropna().drop_duplicates()
    if df.empty:
        print(f"❌ {bank} ({ticker}) — empty after cleaning.")
        continue

    # Save & show
    results[(bank,ticker,source)] = df
    print(f"✅ {bank} ({ticker}) from {source}: {len(df)} rows")
    display(pretty(df))

    save_bank_csv(df, bank, ticker)

# Summary
print("\nSummary:")
for (bank,ticker,source), df in results.items():
    print(f"  {bank} ({ticker}) — {source}: {df['date'].min().date()} → {df['date'].max().date()} ({len(df)} rows)")


✅ JP Morgan (JPM) from yfinance: 5 rows


,date,revenue_total,revenue_total_bil
4,2024-06-30,4.206800e+10,42.068
3,2024-09-30,4.265600e+10,42.656
2,2024-12-31,4.279100e+10,42.791
1,2025-03-31,4.532700e+10,45.327
0,2025-06-30,4.488200e+10,44.882


Saved → C:\Users\User\bootcamp_Khushi_Khanna\project\data\raw\revenue_actuals_manual_JPM_20250825-1335.csv
✅ Deutsche Bank (DB) from yfinance: 5 rows


,date,revenue_total,revenue_total_bil
4,2024-06-30,7.598000e+09,7.598
3,2024-09-30,7.481000e+09,7.481
2,2024-12-31,7.192000e+09,7.192
1,2025-03-31,8.544000e+09,8.544
0,2025-06-30,7.821000e+09,7.821


Saved → C:\Users\User\bootcamp_Khushi_Khanna\project\data\raw\revenue_actuals_manual_DB_20250825-1335.csv
✅ Morgan Stanley (MS) from yfinance: 5 rows


...

Saved → C:\Users\User\bootcamp_Khushi_Khanna\project\data\raw\revenue_actuals_manual_MS_20250825-1335.csv
✅ Bank of America (BAC) from yfinance: 5 rows


,date,revenue_total,revenue_total_bil
4,2024-06-30,2.537700e+10,25.377
3,2024-09-30,2.534500e+10,25.345
2,2024-12-31,2.534700e+10,25.347
1,2025-03-31,2.736600e+10,27.366
0,2025-06-30,2.646300e+10,26.463


Saved → C:\Users\User\bootcamp_Khushi_Khanna\project\data\raw\revenue_actuals_manual_BAC_20250825-1335.csv
✅ Goldman Sachs (GS) from yfinance: 5 rows


,date,revenue_total,revenue_total_bil
4,2024-06-30,1.273100e+10,12.731
3,2024-09-30,1.269900e+10,12.699
2,2024-12-31,1.386900e+10,13.869
1,2025-03-31,1.506200e+10,15.062
0,2025-06-30,1.458300e+10,14.583


Saved → C:\Users\User\bootcamp_Khushi_Khanna\project\data\raw\revenue_actuals_manual_GS_20250825-1335.csv

Summary:
  JP Morgan (JPM) — yfinance: 2024-06-30 → 2025-06-30 (5 rows)
  Deutsche Bank (DB) — yfinance: 2024-06-30 → 2025-06-30 (5 rows)
  Morgan Stanley (MS) — yfinance: 2024-06-30 → 2025-06-30 (5 rows)
  Bank of America (BAC) — yfinance: 2024-06-30 → 2025-06-30 (5 rows)
  Goldman Sachs (GS) — yfinance: 2024-06-30 → 2025-06-30 (5 rows)
